# Session 3 · Understanding HDFS and Its Goals

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hamnakhan123/Diploma-in-Data-Science/blob/main/term-2/02-big-data-systems-with-hadoop/code/session-03-hdfs-and-its-goals.ipynb)

Diploma in Data Science · Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run the cells from top to bottom. Every example is self-contained, so you can also run any single cell on its own.

### Example 2 · A mini NameNode with rack-aware placement

In [1]:
import math, random
random.seed(3)

racks = {"rack1": ["dn1", "dn2", "dn3"], "rack2": ["dn4", "dn5", "dn6"]}
rack_of = {dn: r for r, dns in racks.items() for dn in dns}
BLOCK = 128

def place(writer):
    first = writer
    other_rack = [r for r in racks if r != rack_of[first]][0]
    second = random.choice(racks[other_rack])
    third = random.choice([d for d in racks[other_rack] if d != second])
    return [first, second, third]

namespace = {}          # file -> list of block ids
block_map = {}          # block id -> list of DataNodes
next_id = 1000
for fname, size, writer in [("/data/calls.csv", 400, "dn1"), ("/data/tx.parquet", 200, "dn5")]:
    namespace[fname] = []
    for _ in range(math.ceil(size / BLOCK)):
        block_map[next_id] = place(writer)
        namespace[fname].append(next_id)
        next_id += 1

for f, blocks in namespace.items():
    print(f)
    for b in blocks:
        reps = block_map[b]
        print(f"   blk_{b}: {reps}  racks={sorted({rack_of[d] for d in reps})}")

/data/calls.csv
   blk_1000: ['dn1', 'dn4', 'dn5']  racks=['rack1', 'rack2']
   blk_1001: ['dn1', 'dn5', 'dn6']  racks=['rack1', 'rack2']
   blk_1002: ['dn1', 'dn6', 'dn4']  racks=['rack1', 'rack2']
   blk_1003: ['dn1', 'dn6', 'dn4']  racks=['rack1', 'rack2']
/data/tx.parquet
   blk_1004: ['dn5', 'dn2', 'dn3']  racks=['rack1', 'rack2']
   blk_1005: ['dn5', 'dn3', 'dn1']  racks=['rack1', 'rack2']


### Example 3 · DataNode failure and re-replication

In [2]:
import random
random.seed(11)
nodes = [f"dn{i}" for i in range(1, 7)]
block_map = {f"blk_{i}": random.sample(nodes, 3) for i in range(1, 11)}

def report(title):
    under = {b: r for b, r in block_map.items() if len(r) < 3}
    print(f"{title}: {len(block_map)} blocks, under-replicated = {len(under)}")
    return under

report("healthy cluster")
dead = "dn2"
print(f"\n*** no heartbeat from {dead} for 10 minutes -> marked dead ***")
for reps in block_map.values():
    if dead in reps:
        reps.remove(dead)
under = report("after failure")
for b, reps in sorted(under.items()):
    target = random.choice([n for n in nodes if n != dead and n not in reps])
    print(f"   copy {b} from {reps[0]} to {target}")
    reps.append(target)
report("\nafter re-replication")

healthy cluster: 10 blocks, under-replicated = 0

*** no heartbeat from dn2 for 10 minutes -> marked dead ***
after failure: 10 blocks, under-replicated = 6
   copy blk_10 from dn1 to dn3
   copy blk_2 from dn4 to dn6
   copy blk_3 from dn5 to dn1
   copy blk_4 from dn6 to dn3
   copy blk_6 from dn1 to dn4
   copy blk_8 from dn6 to dn3

after re-replication: 10 blocks, under-replicated = 0


{}

### Example 4 · JobTracker scheduling with data locality

In [3]:
import random
random.seed(5)
nodes = ["tt1", "tt2", "tt3", "tt4"]
blocks = {f"split{i}": random.sample(nodes, 3) for i in range(1, 11)}
slots = {n: 2 for n in nodes}          # 2 map slots per TaskTracker
pending = list(blocks)
assign = []
while pending and any(slots.values()):
    for tt in nodes:
        if not slots[tt] or not pending:
            continue
        local = [s for s in pending if tt in blocks[s]]
        task = local[0] if local else pending[0]
        assign.append((tt, task, "node-local" if local else "remote"))
        pending.remove(task); slots[tt] -= 1

for tt, task, kind in assign:
    print(f"{tt} runs map({task})  [{kind}]")
print(f"\nlocal tasks: {sum(k == 'node-local' for *_, k in assign)}/{len(assign)}, waiting: {pending}")

tt1 runs map(split2)  [node-local]
tt2 runs map(split1)  [node-local]
tt3 runs map(split3)  [node-local]
tt4 runs map(split5)  [node-local]
tt1 runs map(split4)  [node-local]
tt2 runs map(split6)  [node-local]
tt3 runs map(split8)  [node-local]
tt4 runs map(split7)  [node-local]

local tasks: 8/8, waiting: ['split9', 'split10']
